# CrowdHumanでYOLO人物検出モデルをファインチューニング

CrowdHumanの **vbox（人物の見えている範囲）とhbox（頭部）** を使ってYOLOをファインチューニングします。1つのモデルで、`0: person`（vbox）と `1: head`（hbox）の2クラスを学習します。

GPU確認・Google Drive接続・2クラスのデータセット準備まで実装しています。上から順に実行してください。学習処理はまだ追加していません。

学習処理の実装方針:

- 小規模学習は挟まず、設定したエポック数で学習します。
- 各エポック終了時に、再開に必要な状態を含むチェックポイントと学習結果をGoogle Driveへ保存します。
- セッション中断後は、最後に正常に保存できたエポックから再開します。エポック途中からの再開は対象にしません。
- これらの保存・再開処理は、まだ実装していません。

確認済みの保存場所:

- データ: `/content/drive/MyDrive/crowdhuman`
- 注釈: `raw/annotations/annotation_train.odgt`、`raw/annotations/annotation_val.odgt`（データフォルダからの相対パス）
- 保存済みデータの管理情報: `prepared/latest.json`（データフォルダからの相対パス）
- 管理情報の参照先: `prepared_9c497bb835b6435ea77e0992ab70b7d4.tar`（`prepared` 内）。設定は `vbox`。hboxを含むか、tarの中身は未確認です。

今後使用する保存先:

- 学習結果: `/content/drive/MyDrive/crowdhuman_vbox_ft/runs`
- 事前学習重み: `/content/drive/MyDrive/crowdhuman_vbox_ft/weights`
- ローカルデータ展開先: `/content/crowdhuman_vbox_yolo`


## 1. GPUを確認

Colabの「ランタイム」→「ランタイムのタイプを変更」でGPUを選択してから実行してください。
`nvidia-smi` の結果と、PyTorchから見えるGPU・メモリ・バージョンを表示します。


In [ ]:
import subprocess
import torch

try:
    result = subprocess.run(
        ["nvidia-smi"], capture_output=True, text=True, check=False
    )
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
except FileNotFoundError:
    print("nvidia-smiが見つかりません。ColabのGPU設定を確認してください。")

print(f"PyTorch: {torch.__version__}")
print(f"PyTorchのCUDAバージョン: {torch.version.cuda}")
print(f"CUDA利用可能: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for index in range(torch.cuda.device_count()):
        properties = torch.cuda.get_device_properties(index)
        print(f"GPU {index}: {properties.name}")
        print(f"GPU {index}の総メモリ: {properties.total_memory / 1024**3:.2f} GiB")
else:
    print("GPUが利用できません。ランタイムのタイプを確認してください。")


## 2. Google Driveを接続

画面の案内に従って、CrowdHumanを保存しているGoogleアカウントのDriveを接続してください。


In [ ]:
from google.colab import drive

drive.mount("/content/drive")


## 3. データセット準備の設定

Driveの `raw/archives` の4個のZIPと、`raw/annotations` のtrain・val注釈を使用します。
`prepared` のtarは管理情報がvbox設定のため、今回の2クラス変換には使用しません。
Driveの元データは変更しません。ローカルへの展開とラベル変換には時間・空き容量が必要です。

出力先は今回の準備処理専用にしてください。設定が一致する途中出力は再利用します。
Colabのローカルデータが失われた場合は、セル3〜4を再実行して準備し直します。


In [ ]:
from pathlib import Path

DATA_DIR = Path("/content/drive/MyDrive/crowdhuman")
OUTPUT_DIR = Path("/content/crowdhuman_vbox_yolo")

if not Path("/content/drive/MyDrive").is_dir():
    raise RuntimeError("先にセル2でGoogle Driveを接続してください。")


## 4. 画像の展開と2クラスのYOLOラベル生成

元のtrain・val分割を維持し、次のルールで変換します。

- `tag=person` かつ `extra.ignore` が未指定または0の人物を対象にします。
- `0: person` はvbox、`1: head` はhboxです。頭部はさらに `head_attr.ignore` が未指定または0のものを使います。
- 座標 `[x, y, w, h]` を画像内にクリップし、YOLOの正規化座標に変換します。欠損・非有限値・幅高さが0以下・画像内の面積が0の枠は除きます。
- 空ラベルの画像も残します。除外数と各クラスの枠数は処理後に表示し、JSONにも保存します。

**無視枠をラベルから除いても、学習時にその領域の損失が無効になるわけではありません。** この出力は通常のYOLO用データで、CrowdHuman公式の無視領域対応や公式評価の実装ではありません。元注釈はDriveに保持します。
注釈形式の出典: [CrowdHuman公式仕様](https://www.crowdhuman.org/download.html)

ZIPは1個ずつローカルへコピーし、注釈に対応する画像だけ展開します。初回展開時はZIP読み取りのCRCチェックが働き、変換時は全画像をデコードします。同じ設定で再実行すると、サイズが一致する展開済み画像を再利用し、ラベルを再生成します。ZIP全体のハッシュ照合は行いません。
完了後、`images/{train,val}`、`labels/{train,val}`、`data.yaml`、`preparation_report.json` が出力されます。

**完了時の集計結果を共有してください。学習は実行しません。**


In [ ]:
from pathlib import Path
import json
import hashlib
import math
import shutil
import tempfile
import zipfile
from collections import Counter
from PIL import Image
from tqdm.auto import tqdm


def read_records(path):
    with path.open(encoding="utf-8") as stream:
        for line_number, line in enumerate(stream, 1):
            if not line.strip():
                continue
            record = json.loads(line)
            image_id = record.get("ID")
            if (not isinstance(image_id, str) or not image_id
                    or "/" in image_id or "\\" in image_id or image_id in (".", "..")):
                raise ValueError(f"不正な画像ID: {path}:{line_number}")
            if not isinstance(record.get("gtboxes"), list):
                raise ValueError(f"gtboxesがありません: {path}:{line_number}")
            yield record


def normalized_box(box, width, height):
    if not isinstance(box, (list, tuple)) or len(box) != 4:
        return None
    try:
        x, y, w, h = map(float, box)
    except (TypeError, ValueError):
        return None
    if not all(math.isfinite(v) for v in (x, y, w, h)) or w <= 0 or h <= 0:
        return None
    left, top = max(0.0, x), max(0.0, y)
    right, bottom = min(float(width), x + w), min(float(height), y + h)
    if right <= left or bottom <= top:
        return None
    return ((left + right) / (2 * width), (top + bottom) / (2 * height),
            (right - left) / width, (bottom - top) / height)


def labels_for_record(record, width, height):
    lines, counts = [], Counter()
    for obj in record["gtboxes"]:
        if obj.get("tag") != "person":
            counts["excluded_non_person"] += 1
            continue
        if (obj.get("extra") or {}).get("ignore", 0) != 0:
            counts["excluded_person_ignore"] += 1
            continue
        for class_id, key in ((0, "vbox"), (1, "hbox")):
            if class_id == 1 and (obj.get("head_attr") or {}).get("ignore", 0) != 0:
                counts["excluded_head_ignore"] += 1
                continue
            box = normalized_box(obj.get(key), width, height)
            if box is None:
                counts[f"invalid_{key}"] += 1
                continue
            lines.append(f"{class_id} " + " ".join(f"{value:.10f}" for value in box))
            counts[key] += 1
    return lines, counts


def atomic_text(path, text):
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(text, encoding="utf-8")
    temporary.replace(path)


def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def prepare_dataset(data_dir, output_dir):
    annotation_paths = {
        split: data_dir / "raw" / "annotations" / f"annotation_{split}.odgt"
        for split in ("train", "val")
    }
    archive_paths = {
        "train": [data_dir / "raw" / "archives" / f"CrowdHuman_train{i:02d}.zip"
                  for i in (1, 2, 3)],
        "val": [data_dir / "raw" / "archives" / "CrowdHuman_val.zip"],
    }
    source_paths = list(annotation_paths.values()) + sum(archive_paths.values(), [])
    for path in source_paths:
        if not path.is_file() or path.stat().st_size == 0:
            raise FileNotFoundError(f"必要なファイルがないか空です: {path}")

    config = {
        "schema": 1,
        "classes": {"0": "person_vbox", "1": "head_hbox"},
        "policy": "person_tag; extra.ignore=0; head_attr.ignore=0; clip; discard_invalid",
        "annotations": {split: sha256_file(path) for split, path in annotation_paths.items()},
        # ZIPはサイズと更新時刻を記録します。全体のハッシュ検証ではありません。
        "archives": {str(path): {"size": path.stat().st_size, "mtime_ns": path.stat().st_mtime_ns}
                     for path in sum(archive_paths.values(), [])},
    }
    marker = output_dir / ".build_config.json"
    if output_dir.exists() and any(output_dir.iterdir()):
        if not marker.is_file() or json.loads(marker.read_text()) != config:
            raise RuntimeError("出力先に別設定または管理対象外のデータがあります。OUTPUT_DIRを別の空フォルダに変更してください。")
    output_dir.mkdir(parents=True, exist_ok=True)
    atomic_text(marker, json.dumps(config, ensure_ascii=False, indent=2))

    ids = {}
    for split, path in annotation_paths.items():
        ids[split] = set()
        for record in read_records(path):
            if record["ID"] in ids[split]:
                raise ValueError(f"注釈の画像ID重複: {split}/{record['ID']}")
            ids[split].add(record["ID"])
        if not ids[split]:
            raise ValueError(f"注釈が空です: {path}")
        for kind in ("images", "labels"):
            (output_dir / kind / split).mkdir(parents=True, exist_ok=True)
    if ids["train"] & ids["val"]:
        raise ValueError("trainとvalで画像IDが重複しています。")

    # ZIPの目録を照合します。全画像の展開前に欠落・重複を検出します。
    plans, required_bytes, largest_zip = [], 0, 0
    for split, paths in archive_paths.items():
        found = set()
        for path in paths:
            entries = []
            with zipfile.ZipFile(path) as archive:
                for member in archive.infolist():
                    name = Path(member.filename)
                    if member.is_dir() or name.suffix.lower() != ".jpg" or name.stem not in ids[split]:
                        continue
                    if name.stem in found:
                        raise ValueError(f"ZIP内の画像ID重複: {split}/{name.stem}")
                    found.add(name.stem)
                    target = output_dir / "images" / split / f"{name.stem}.jpg"
                    if not target.is_file() or target.stat().st_size != member.file_size:
                        required_bytes += member.file_size
                    entries.append((member.filename, target, member.file_size))
            plans.append((path, entries))
            largest_zip = max(largest_zip, path.stat().st_size)
        missing = ids[split] - found
        if missing:
            raise ValueError(f"{split}: ZIPに画像が{len(missing)}件不足。例: {sorted(missing)[:5]}")
        print(f"{split}: 注釈とZIPの画像ID一致 {len(found):,}件")

    free = shutil.disk_usage(output_dir).free
    needed = required_bytes + largest_zip + 2 * 1024**3
    print(f"ローカル空き: {free / 1024**3:.2f} GiB / 必要目安: {needed / 1024**3:.2f} GiB")
    if free < needed:
        raise RuntimeError("ローカル空き容量が不足しています。Driveの元データは変更していません。")

    # 一度に1個だけZIPをローカルへコピーし、必要な画像だけ取り出します。
    # 一時領域のみ自動削除。出力画像は一時ファイル経由で確定します。
    with tempfile.TemporaryDirectory(prefix="crowdhuman_zip_", dir=output_dir.parent) as cache:
        for source, entries in plans:
            pending = [(name, target, size) for name, target, size in entries
                       if not target.is_file() or target.stat().st_size != size]
            if not pending:
                print(f"{source.name}: 展開済みのためスキップ")
                continue
            local_zip = Path(cache) / source.name
            print(f"ローカルへコピー中: {source.name}")
            shutil.copyfile(source, local_zip)
            with zipfile.ZipFile(local_zip) as archive:
                for name, target, size in tqdm(pending, desc=source.name):
                    temporary = target.with_name(target.name + ".part")
                    with archive.open(name) as src, temporary.open("wb") as dst:
                        shutil.copyfileobj(src, dst, length=1024 * 1024)
                    if temporary.stat().st_size != size:
                        raise IOError(f"画像サイズ不一致: {name}")
                    temporary.replace(target)
            local_zip.unlink()

    summary = {}
    for split, annotation_path in annotation_paths.items():
        counts = Counter()
        for record in tqdm(read_records(annotation_path), total=len(ids[split]), desc=f"{split} ラベル変換"):
            image_path = output_dir / "images" / split / f"{record['ID']}.jpg"
            # 全画像をデコードし、破損や読み取り不能を検出します。
            with Image.open(image_path) as image:
                image.load()
                width, height = image.size
            lines, box_counts = labels_for_record(record, width, height)
            counts.update(box_counts)
            counts["images"] += 1
            counts["empty_labels"] += int(not lines)
            label_path = output_dir / "labels" / split / f"{record['ID']}.txt"
            atomic_text(label_path, "\n".join(lines) + ("\n" if lines else ""))
        summary[split] = dict(counts)
        if counts["vbox"] == 0 or counts["hbox"] == 0:
            raise ValueError(f"{split}: personまたはheadの有効ラベルが0件です。")
        if len(list((output_dir / "images" / split).glob("*.jpg"))) != len(ids[split]):
            raise ValueError(f"{split}: 出力先に余分な画像があります。別の空フォルダを指定してください。")
        if len(list((output_dir / "labels" / split).glob("*.txt"))) != len(ids[split]):
            raise ValueError(f"{split}: 出力ラベル数が一致しません。")

    yaml_path = output_dir / "data.yaml"
    yaml_text = (f"path: {json.dumps(str(output_dir.resolve()))}\n"
                 "train: images/train\nval: images/val\nnames:\n  0: person\n  1: head\n")
    atomic_text(yaml_path, yaml_text)
    atomic_text(output_dir / "preparation_report.json", json.dumps(summary, ensure_ascii=False, indent=2))
    print(json.dumps(summary, ensure_ascii=False, indent=2))
    print(f"準備完了。学習用YAML: {yaml_path}")
    return yaml_path

DATA_YAML = prepare_dataset(DATA_DIR, OUTPUT_DIR)
